# Model Evaluation with LMEval

Evaluate the quality impact of INT8 quantization by comparing original and compressed models.

### 1. Install and import libraries

In [ ]:
import os
from pathlib import Path
from lm_eval import simple_evaluate

print("✓ Libraries ready")

### 2. Configure evaluation

In [ ]:
ORIGINAL_MODEL = "/shared-models/granite-4.0-350m"
COMPRESSED_MODEL = "../compression/granite-INT8"
TASK = "arc_easy"
LIMIT = 50  # Number of samples (for faster evaluation)
BATCH_SIZE = "64"

# Verify compressed model exists
if not Path(COMPRESSED_MODEL).exists():
    raise FileNotFoundError("Compressed model not found. Run the compression notebook first.")

### 3. Evaluate original model

In [ ]:
results_orig = simple_evaluate(
    model="hf",
    model_args=f"pretrained={ORIGINAL_MODEL},dtype=auto",
    tasks=[TASK],
    device="cpu",
    batch_size=BATCH_SIZE,
    limit=LIMIT
)

print("✓ Original model evaluation complete")
print(f"Accuracy: {results_orig['results'][TASK]['acc,none'] * 100:.2f}%")

### 4. Evaluate compressed model

In [ ]:
# Load compressed model with transformers (supports compressed-tensors)
from transformers import AutoModelForCausalLM, AutoTokenizer
from lm_eval.models.huggingface import HFLM

compressed_path = os.path.abspath(COMPRESSED_MODEL)

# Load model and tokenizer
print("Loading compressed model...")
model = AutoModelForCausalLM.from_pretrained(
    compressed_path, 
    torch_dtype="auto", 
    low_cpu_mem_usage=True
)
tokenizer = AutoTokenizer.from_pretrained(compressed_path)

# Wrap in lm-eval's HFLM wrapper
print("Preparing model for evaluation...")
lm = HFLM(
    pretrained=model,
    tokenizer=tokenizer,
    batch_size=BATCH_SIZE
)

# Evaluate using the model object
print("Evaluating...")
results_comp = simple_evaluate(
    model=lm,
    tasks=[TASK],
    limit=LIMIT
)

print("✓ Compressed model evaluation complete")
print(f"Accuracy: {results_comp['results'][TASK]['acc,none'] * 100:.2f}%")

### 5. Compare results

In [ ]:
# Extract accuracy
orig_acc = results_orig['results'][TASK]['acc,none'] * 100
comp_acc = results_comp['results'][TASK]['acc,none'] * 100
diff = comp_acc - orig_acc

# Display comparison
print("="*60)
print("Quality Impact of INT8 Quantization")
print("="*60)
print(f"Benchmark:       {TASK}")
print(f"Original:        {orig_acc:>6.2f}%")
print(f"Compressed:      {comp_acc:>6.2f}%")
print(f"Difference:      {diff:>+6.2f}%")
print("="*60)

if abs(diff) < 2:
    print("✓ Minimal quality loss - excellent compression")
elif abs(diff) < 5:
    print("⚠ Acceptable quality trade-off")
else:
    print("✗ Significant quality impact - consider different strategy")